# 11 · Explaining Single Predictions

**Goal.** For individual posts, show **which words pushed the model toward its decision**.

**Why it matters.** Trust requires transparency. We use three independent methods (**SHAP, LIME, and
Integrated Gradients**) and check whether they agree.

**What happens below.** Explain a set of correct and incorrect predictions, and verify the explanations
faithfully reflect the real model.

**Output:** `data/stage4_explain/local_explanations.jsonl` + figures.

## 0. Environment & dependencies

In [ ]:
import os, sys, json, time, math, logging, warnings, random
from pathlib import Path
from collections import Counter, defaultdict

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

assert _try("torch") and _try("transformers") and _try("sklearn"), "notebook 11 needs torch+transformers+sklearn"
HAS_MPL = _try("matplotlib")
# D1: no XAI library is required/used — SHAP/LIME/IG are implemented from scratch below.
for _lib in ("shap", "lime", "captum"):
    if _try(_lib):
        print(f"note: {_lib} is installed but NOT used (from-scratch impls per D1, for reproducibility).")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("xai")
PART_VERSION = "1.0"
SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 1. Configuration (D3 instance set · D4 IG steps · sample budgets)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED)

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage3_model").exists() or (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR   = PROJECT_ROOT / "data" / "stage2_semantic"
TEMP_DIR  = PROJECT_ROOT / "data" / "stage1_temporal"
MODEL_DIR = PROJECT_ROOT / "data" / "stage3_model"
OUTPUT_DIR = PROJECT_ROOT / "data" / "stage4_explain"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"   # CPU is sufficient and deterministic for the bounded instance set
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}

# ---- backbones (must match Objective 3 / notebook 06 exactly) ----
MURIL_ID = "google/muril-base-cased"
MURIL_MAX_TOKENS = 128          # notebook 06 setting (anchor_similarity_report: max_tokens=128, mean pool)
EMO_ID = "MilaNLProc/xlm-emo-t"
EMO_MAX_TOKENS = 128
EMBED_BATCH = 32

# ---- D3: curated, logged instance set ----
N_TOP_CORRECT_PER_TIER = 5      # top-confidence CORRECT predictions per Tier
N_ERRORS = 8                    # most-confident ERRORS (pred != true)
MAX_WORDS = 40                  # cap attributed words per instance (bounds SHAP combinatorics); logged if truncated

# ---- attribution budgets ----
SHAP_SAMPLES = 160              # KernelSHAP coalitions per instance (+ empty/full)
LIME_SAMPLES = 160              # LIME perturbations per instance
LIME_KERNEL_WIDTH = 0.25        # exponential kernel width on cosine distance
LIME_RIDGE_ALPHA = 0.01
IG_STEPS = 32                   # D4: Riemann steps, zero-embedding baseline
TOPK = 6                        # top-k trigger words for agreement
N_HEATMAP_FIGS = 3             # per-instance attribution figures to render

log.info("instances: top%d correct/tier + %d errors | SHAP=%d LIME=%d IG_steps=%d | topk=%d",
         N_TOP_CORRECT_PER_TIER, N_ERRORS, SHAP_SAMPLES, LIME_SAMPLES, IG_STEPS, TOPK)

## 2. Re-define the Objective 3 model verbatim (so we explain the *actual* deployed network)

The `DriftCapsNet` (+ `CapsuleRouting`, `squash`) below is **byte-for-byte the notebook 09 definition**. We load
`model_state.pt` into it with the saved config (static routing / ELU gate / attention fusion). This is the real
network whose decisions we explain — not a surrogate (carry-in invariant 1).

In [ ]:
CFG = json.loads((MODEL_DIR / "model_config.json").read_text(encoding="utf-8"))
FUSION_DIM = CFG["fusion_dim"]; PRIM_CAPS = CFG["prim_caps"]; PRIM_DIM = CFG["prim_dim"]
CLASS_DIM = CFG["class_dim"]; N_CLASSES = 4
ROUTING = CFG["routing"]; GATE = CFG["gate"]; FUSION = CFG["fusion"]; DROPOUT = 0.3
TEMP_FEATS = CFG["temp_feats"]

def squash(s, dim=-1, eps=1e-8):
    n2 = (s * s).sum(dim=dim, keepdim=True)
    return (n2 / (1.0 + n2)) * (s / torch.sqrt(n2 + eps))

class CapsuleRouting(nn.Module):
    def __init__(self, n_in, d_in, n_out, d_out, routing="static", iters=3):
        super().__init__()
        self.n_in, self.n_out, self.routing, self.iters = n_in, n_out, routing, iters
        self.W = nn.Parameter(0.01 * torch.randn(1, n_in, n_out, d_out, d_in))
        self.b_static = nn.Parameter(torch.zeros(1, n_in, n_out))

    def forward(self, u):
        B = u.size(0)
        u_hat = torch.matmul(self.W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        if self.routing == "static":
            c = torch.softmax(self.b_static, dim=2).unsqueeze(-1)
            s = (c * u_hat).sum(dim=1)
            return squash(s)
        b = torch.zeros(B, self.n_in, self.n_out, device=u.device)
        for it in range(self.iters):
            c = torch.softmax(b, dim=2).unsqueeze(-1)
            s = (c * u_hat).sum(dim=1)
            v = squash(s)
            if it < self.iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(-1)
        return v

class DriftCapsNet(nn.Module):
    def __init__(self, text_dim, emo_dim, temp_dim=6, d=FUSION_DIM, gate=GATE, fusion=FUSION,
                 routing=ROUTING, dropout=DROPOUT):
        super().__init__()
        self.gate_kind, self.fusion = gate, fusion
        self.text_proj = nn.Linear(text_dim, PRIM_CAPS * PRIM_DIM)
        self.gate_lin  = nn.Linear(PRIM_CAPS * PRIM_DIM, PRIM_CAPS * PRIM_DIM)
        self.caps = CapsuleRouting(PRIM_CAPS, PRIM_DIM, N_CLASSES, CLASS_DIM, routing=routing)
        self.q_proj = nn.Linear(N_CLASSES * CLASS_DIM, d)
        self.e_proj = nn.Linear(emo_dim, d)
        self.t_proj = nn.Sequential(nn.Linear(temp_dim, 64), nn.ReLU(), nn.Linear(64, d))
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d)
        head_in = d if fusion in ("attention", "text_only") else 3 * d
        self.head = nn.Linear(head_in, N_CLASSES)
        self.d = d

    def gate(self, H):
        if self.gate_kind == "none":   return H
        if self.gate_kind == "sigmoid":return H * torch.sigmoid(self.gate_lin(H))
        return H * (F.elu(self.gate_lin(H)) + 1.0)

    def temporal_rep(self, x_t, mask):
        return self.t_proj(x_t) * mask.unsqueeze(-1)

    def forward(self, x_text, x_emo, x_temp, mask, return_z=False):
        H = self.drop(F.elu(self.text_proj(x_text)))
        H_g = self.gate(H)
        U = squash(H_g.view(-1, PRIM_CAPS, PRIM_DIM))
        V = self.caps(U).reshape(x_text.size(0), -1)
        q = self.q_proj(V)
        e = self.e_proj(x_emo)
        tau = self.temporal_rep(x_temp, mask)
        if self.fusion == "attention":
            kv = torch.stack([tau, e], dim=1)
            att = torch.softmax(torch.matmul(q.unsqueeze(1), kv.transpose(1, 2)) / math.sqrt(self.d), dim=-1)
            ctx = torch.matmul(att, kv).squeeze(1)
            z = self.norm(q + ctx)
        elif self.fusion == "concat":
            z = torch.cat([q, tau, e], dim=-1)
        else:
            z = q
        logits = self.head(z)
        return (logits, F.normalize(z, dim=-1)) if return_z else logits

model = DriftCapsNet(CFG["text_dim"], CFG["emo_dim"]).to(DEVICE)
state = torch.load(MODEL_DIR / "model_state.pt", map_location=DEVICE)
model.load_state_dict(state)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
log.info("loaded trained Objective 3 model (%s routing / %s gate / %s fusion)", ROUTING, GATE, FUSION)

## 3. Load frozen MuRIL + emotion backbones (live — we re-embed perturbed text)

Unlike Objective 3 (which used cached embeddings), local XAI must re-embed *perturbed* texts, so both text-path
backbones are loaded live and frozen. Mean-pool @128 reproduces the notebook 06 representation.

In [ ]:
from transformers import AutoTokenizer, AutoModel

mtok = AutoTokenizer.from_pretrained(MURIL_ID)
mmodel = AutoModel.from_pretrained(MURIL_ID).to(DEVICE).eval()
etok = AutoTokenizer.from_pretrained(EMO_ID)
emodel = AutoModel.from_pretrained(EMO_ID).to(DEVICE).eval()
for p in list(mmodel.parameters()) + list(emodel.parameters()):
    p.requires_grad_(False)

def _mean_pool(hs, mask):
    m = mask.unsqueeze(-1).float()
    return (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def muril_embed(texts):
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        chunk = [t if t.strip() else "[UNK]" for t in texts[i:i+EMBED_BATCH]]
        enc = mtok(chunk, padding=True, truncation=True, max_length=MURIL_MAX_TOKENS, return_tensors="pt").to(DEVICE)
        out.append(_mean_pool(mmodel(**enc).last_hidden_state, enc["attention_mask"]).cpu().numpy())
    return np.vstack(out).astype(np.float32)

@torch.no_grad()
def emo_embed(texts):
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        chunk = [t if t.strip() else "[UNK]" for t in texts[i:i+EMBED_BATCH]]
        enc = etok(chunk, padding=True, truncation=True, max_length=EMO_MAX_TOKENS, return_tensors="pt").to(DEVICE)
        out.append(_mean_pool(emodel(**enc).last_hidden_state, enc["attention_mask"]).cpu().numpy())
    return np.vstack(out).astype(np.float32)

log.info("MuRIL + %s loaded (frozen)", EMO_ID)

## 4. Load TEST corpus + predictions + temporal; assert f reproduces the stored embedding

Sanity (carry-in invariant 1): re-embedding the original `text_muril` through our live MuRIL routine must
reproduce the notebook 06 stored `embeddings_test.npy` row (cosine > 0.999). If this fails, `f` is not the
deployed function and every attribution below would be meaningless.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
assert set(r["platform"] for r in test_rows) <= {"reddit", "twitter"}, "TEST platform leak!"
by_id = {r["record_id"]: r for r in test_rows}

preds = load_jsonl(MODEL_DIR / "predictions_test.jsonl")
pred_by_id = {r["record_id"]: r for r in preds}

temp_rows = load_jsonl(TEMP_DIR / "temporal_features_test.jsonl")
temp_by_id = {r["record_id"]: r for r in temp_rows}

def temp_vec(rid):
    t = temp_by_id.get(rid, {})
    feats = np.array([[float(t.get(k, 0.0)) for k in TEMP_FEATS]], np.float32)
    mask = np.array([float(t.get("temporal_mask", by_id[rid].get("temporal_mask", 1)))], np.float32)
    return feats, mask

def text_of(rid):
    r = by_id[rid]
    return str(r.get("text_muril") or r.get("post_text_clean") or "")

# --- faithfulness assertion ---
stored = np.load(SEM_DIR / "embeddings_test.npy").astype(np.float32)
idx = json.loads((SEM_DIR / "embeddings_index_test.json").read_text(encoding="utf-8"))
id2row = {rid: i for i, rid in enumerate(idx)}
probe_ids = [test_rows[i]["record_id"] for i in range(0, min(len(test_rows), 30), 6)]
re_emb = muril_embed([text_of(r) for r in probe_ids])
cos = []
for k, rid in enumerate(probe_ids):
    a = re_emb[k]; b = stored[id2row[rid]]
    cos.append(float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)))
min_cos = float(np.min(cos))
log.info("faithfulness: re-embed vs stored MuRIL cosine min=%.5f mean=%.5f over %d probes",
         min_cos, float(np.mean(cos)), len(cos))
assert min_cos > 0.999, f"f does NOT reproduce the deployed embedding (min cos {min_cos:.5f}) — abort"
print(f"faithfulness OK: min cosine {min_cos:.5f} (f reproduces the deployed MuRIL representation)")

## 5. The deployed scalar function f(text) -> P(target tier)

`f` re-embeds a list of texts through frozen MuRIL **and** frozen emotion (faithful to the full text path),
holds the instance's temporal features/mask fixed, runs the trained head, and returns the softmax probability
of a chosen `target` tier. This is the function SHAP and LIME perturb.

In [ ]:
@torch.no_grad()
def f_probs(texts, temp_feat, temp_mask):
    # texts -> (N,4) softmax over tiers; temporal features/mask fixed (broadcast over the batch)
    mx = torch.tensor(muril_embed(texts))
    ex = torch.tensor(emo_embed(texts))
    n = mx.size(0)
    tf = torch.tensor(np.repeat(temp_feat, n, axis=0))
    tm = torch.tensor(np.repeat(temp_mask, n, axis=0))
    logits = model(mx.to(DEVICE), ex.to(DEVICE), tf.to(DEVICE), tm.to(DEVICE))
    return torch.softmax(logits, dim=-1).cpu().numpy()

def f_target(texts, temp_feat, temp_mask, target):
    return f_probs(texts, temp_feat, temp_mask)[:, target]

def sig(x, n=6):
    # round to n SIGNIFICANT figures (preserves small IG magnitudes ~1e-9; fixed-decimal rounding would zero them)
    x = float(x)
    if x == 0.0 or not math.isfinite(x):
        return 0.0
    return round(x, -int(math.floor(math.log10(abs(x)))) + (n - 1))

def words_of(rid):
    w = text_of(rid).split()
    truncated = len(w) > MAX_WORDS
    return w[:MAX_WORDS], truncated

def join_mask(words, z):
    return " ".join(w for w, keep in zip(words, z) if keep)

# quick check the function runs and matches the saved prediction for one instance
_rid0 = test_rows[0]["record_id"]
_tf0, _tm0 = temp_vec(_rid0)
_p0 = f_probs([text_of(_rid0)], _tf0, _tm0)[0]
log.info("f sanity: rid=%s argmax=%d (saved pred=%d)", _rid0, int(_p0.argmax()),
         pred_by_id[_rid0]["tier_pred"])

## 6. Select the curated, logged instance set

Top-confidence **correct** predictions per Tier + the most-confident **errors**. Each instance records its
true/predicted Tier, confidence, word count, and whether it was word-truncated — full provenance (P-series honesty).

In [ ]:
rank = []
for r in preds:
    rid = r["record_id"]
    if rid not in by_id:
        continue
    conf = float(max(r["probs"]))
    rank.append((rid, int(r["tier_true"]), int(r["tier_pred"]), conf))

chosen, seen = [], set()
for t in range(4):
    correct = sorted([x for x in rank if x[1] == t and x[1] == x[2]], key=lambda x: -x[3])
    for x in correct[:N_TOP_CORRECT_PER_TIER]:
        if x[0] not in seen and len(text_of(x[0]).split()) >= 2:
            chosen.append(("correct", *x)); seen.add(x[0])
errors = sorted([x for x in rank if x[1] != x[2]], key=lambda x: -x[3])
ne = 0
for x in errors:
    if x[0] not in seen and len(text_of(x[0]).split()) >= 2:
        chosen.append(("error", *x)); seen.add(x[0]); ne += 1
        if ne >= N_ERRORS:
            break

log.info("selected %d instances: %d correct + %d errors", len(chosen),
         sum(1 for c in chosen if c[0] == "correct"), sum(1 for c in chosen if c[0] == "error"))
for kind, rid, yt, yp, conf in chosen:
    log.info("  [%s] rid=%s true=%s pred=%s conf=%.3f nwords=%d", kind, rid,
             TIER_NAMES[yt], TIER_NAMES[yp], conf, len(text_of(rid).split()))

## 7. Method 1 — SHAP (sampling KernelSHAP over word presence, from scratch · D1)

Shapley values via the kernel-weighted linear form (Lundberg & Lee 2017). For a text of `M` words we sample
coalitions `z in {0,1}^M`, evaluate `f` on the masked text, and solve the **SHAP-kernel-weighted least squares**
for per-word contributions `phi`, enforcing efficiency `sum phi = f(full) - f(empty)` by eliminating one
variable. `phi_j` is word `j`'s signed contribution to `P(target)`.

In [ ]:
def shap_kernel_weight(m, M):
    # SHAP kernel: (M-1) / [ C(M,m) * m * (M-m) ] ; m = coalition size (0<m<M)
    if m == 0 or m == M:
        return 1e6  # empty/full pinned (handled as constraints / base)
    return (M - 1.0) / (math.comb(M, m) * m * (M - m))

def kernel_shap(words, temp_feat, temp_mask, target, n_samples=SHAP_SAMPLES, rng=None):
    rng = rng or np.random.default_rng(SEED)
    M = len(words)
    f_full = float(f_target([" ".join(words)], temp_feat, temp_mask, target)[0])
    f_empty = float(f_target([""], temp_feat, temp_mask, target)[0])
    if M == 1:
        return np.array([f_full - f_empty], np.float32), f_full, f_empty
    # sample coalitions (sizes drawn proportional to kernel mass), evaluate f in batch
    sizes = np.arange(1, M)
    pk = np.array([shap_kernel_weight(int(s), M) * math.comb(M, int(s)) for s in sizes], np.float64)
    pk = pk / pk.sum()
    Z, W, texts = [], [], []
    for _ in range(n_samples):
        m = int(rng.choice(sizes, p=pk))
        idx = rng.choice(M, size=m, replace=False)
        z = np.zeros(M, np.float32); z[idx] = 1.0
        Z.append(z); W.append(shap_kernel_weight(m, M)); texts.append(join_mask(words, z))
    Z = np.array(Z, np.float64); W = np.array(W, np.float64)
    fz = f_target(texts, temp_feat, temp_mask, target).astype(np.float64)
    # enforce efficiency: phi_{M-1} = (f_full - f_empty) - sum_{j<M-1} phi_j
    # regress (fz - f_empty - (f_full-f_empty)*z[:,M-1]) on (z[:,:M-1] - z[:,M-1:])
    y = fz - f_empty - (f_full - f_empty) * Z[:, -1]
    X = Z[:, :-1] - Z[:, -1:].repeat(M - 1, axis=1)
    sw = np.sqrt(W)
    Xw = X * sw[:, None]; yw = y * sw
    # ridge-stabilised normal equations
    A = Xw.T @ Xw + 1e-6 * np.eye(M - 1)
    phi_head = np.linalg.solve(A, Xw.T @ yw)
    phi_last = (f_full - f_empty) - phi_head.sum()
    phi = np.concatenate([phi_head, [phi_last]]).astype(np.float32)
    return phi, f_full, f_empty

log.info("KernelSHAP defined")

## 8. Method 2 — LIME (exponential-kernel-weighted sparse linear surrogate, from scratch · D1)

Ribeiro et al. 2016: perturb by randomly dropping words (`z ~ Bernoulli(0.5)`), weight each perturbation by an
**exponential kernel on the cosine distance** between `z` and the all-present vector, and fit a weighted **ridge**
surrogate of `f` on `z`. Surrogate coefficients are the LIME word attributions.

In [ ]:
from sklearn.linear_model import Ridge

def lime_explain(words, temp_feat, temp_mask, target, n_samples=LIME_SAMPLES, rng=None):
    rng = rng or np.random.default_rng(SEED + 1)
    M = len(words)
    ones = np.ones(M)
    Z, texts = [], []
    for _ in range(n_samples):
        z = (rng.random(M) > 0.5).astype(np.float32)
        if z.sum() == 0:
            z[rng.integers(M)] = 1.0
        Z.append(z); texts.append(join_mask(words, z))
    Z = np.array(Z, np.float64)
    fz = f_target(texts, temp_feat, temp_mask, target).astype(np.float64)
    # cosine distance to the all-present instance, then exponential kernel
    num = Z @ ones
    den = (np.linalg.norm(Z, axis=1) * np.linalg.norm(ones) + 1e-9)
    cos = num / den
    dist = 1.0 - cos
    w = np.exp(-(dist ** 2) / (LIME_KERNEL_WIDTH ** 2))
    surrogate = Ridge(alpha=LIME_RIDGE_ALPHA, fit_intercept=True)
    surrogate.fit(Z, fz, sample_weight=w)
    return surrogate.coef_.astype(np.float32)

log.info("LIME defined")

## 9. Method 3 — Integrated Gradients through frozen MuRIL token embeddings (from scratch · D4)

Sundararajan et al. 2017. We tokenize the pre-split words (`is_split_into_words=True`), take MuRIL input-token
embeddings, and integrate the gradient of the **predicted-tier logit** along the straight path from a
**zero-embedding baseline** to the actual embeddings (`IG_STEPS=32`). Emotion + temporal are held fixed (IG
attributes the MuRIL sub-path, D2). Per-subword scores `(emb-baseline)·∫grad` are summed back to **words** via
`word_ids`.

In [ ]:
def ig_explain(words, temp_feat, temp_mask, target):
    enc = mtok(words, is_split_into_words=True, return_tensors="pt",
               truncation=True, max_length=MURIL_MAX_TOKENS)
    input_ids = enc["input_ids"].to(DEVICE)
    attn = enc["attention_mask"].to(DEVICE)
    word_ids = enc.word_ids(0)
    emb_layer = mmodel.get_input_embeddings()
    with torch.no_grad():
        emb_full = emb_layer(input_ids)                       # (1,T,768)
    baseline = torch.zeros_like(emb_full)                     # D4: zero-embedding baseline
    ex = torch.tensor(emo_embed([" ".join(words)])).to(DEVICE)   # emotion fixed
    tf = torch.tensor(temp_feat).to(DEVICE); tm = torch.tensor(temp_mask).to(DEVICE)
    total_grad = torch.zeros_like(emb_full)
    for step in range(1, IG_STEPS + 1):
        alpha = step / IG_STEPS
        emb_a = (baseline + alpha * (emb_full - baseline)).clone().requires_grad_(True)
        hs = mmodel(inputs_embeds=emb_a, attention_mask=attn).last_hidden_state
        mvec = _mean_pool(hs, attn)
        logits = model(mvec, ex, tf, tm)
        score = logits[0, target]
        grad = torch.autograd.grad(score, emb_a)[0]
        total_grad = total_grad + grad.detach()
    avg_grad = total_grad / IG_STEPS
    ig_tok = ((emb_full - baseline) * avg_grad).sum(-1).squeeze(0).cpu().numpy()  # per subword
    # aggregate subwords -> words
    M = len(words)
    word_attr = np.zeros(M, np.float32)
    for ti, wid in enumerate(word_ids):
        if wid is not None and wid < M:
            word_attr[wid] += float(ig_tok[ti])
    return word_attr

log.info("Integrated Gradients defined")

## 10. Run all three methods on every selected instance + per-method top-k triggers

In [ ]:
def topk_words(words, attr, k=TOPK):
    order = np.argsort(-np.abs(attr))[:k]
    return [{"word": words[i], "attr": sig(attr[i])} for i in order]

local_records = []
t0 = time.time()
for kind, rid, yt, yp, conf in chosen:
    words, truncated = words_of(rid)
    tf, tm = temp_vec(rid)
    target = yp  # explain the model's OWN decision (predicted tier)
    shap_attr, f_full, f_empty = kernel_shap(words, tf, tm, target)
    lime_attr = lime_explain(words, tf, tm, target)
    ig_attr = ig_explain(words, tf, tm, target)
    rec = {
        "record_id": rid, "kind": kind, "tier_true": yt, "tier_pred": yp,
        "target_explained": target, "pred_confidence": round(conf, 4),
        "n_words": len(words), "word_truncated": truncated,
        "f_full": round(float(f_full), 4), "f_empty": round(float(f_empty), 4),
        "words": words,
        "shap": [sig(x) for x in shap_attr],
        "lime": [sig(x) for x in lime_attr],
        "ig":   [sig(x) for x in ig_attr],
        "ig_note": "IG magnitudes are small in absolute scale (~1e-9: gradients through the frozen "
                   "mean-pooled MuRIL backbone); the per-instance RANKING is what feeds top-k triggers + "
                   "agreement (fp32==fp64 verified, so this is real signal, not numerical noise).",
        "topk": {
            "shap": topk_words(words, shap_attr),
            "lime": topk_words(words, lime_attr),
            "ig":   topk_words(words, ig_attr),
        },
    }
    local_records.append(rec)
    log.info("  explained rid=%s (%s, target=%s): SHAP top=%s", rid, kind, TIER_NAMES[target],
             [t["word"] for t in rec["topk"]["shap"][:3]])
elapsed = round(time.time() - t0, 1)
log.info("explained %d instances in %.1fs", len(local_records), elapsed)

## 11. Cross-method agreement — top-k Jaccard + Spearman of |attribution|

For each instance: pairwise + 3-way **Jaccard** of the top-k trigger sets, and **Spearman rank correlation** of
the per-word `|attribution|` vectors (all three are word-aligned). Aggregated (mean ± sd) across instances and
broken down correct-vs-error. High agreement across three independent methods is itself one of the expert-free
validity proxies (carried into notebook 12, ).

In [ ]:
def _spear(a, b):
    try:
        from scipy.stats import spearmanr
        r = spearmanr(a, b).correlation
        return float(r) if r == r else 0.0
    except Exception:
        # rank-correlation fallback (no scipy)
        ar = np.argsort(np.argsort(a)); br = np.argsort(np.argsort(b))
        ar = ar - ar.mean(); br = br - br.mean()
        d = (np.linalg.norm(ar) * np.linalg.norm(br) + 1e-9)
        return float((ar @ br) / d)

def jaccard(s1, s2):
    s1, s2 = set(s1), set(s2)
    u = s1 | s2
    return len(s1 & s2) / len(u) if u else 0.0

pairs = [("shap", "lime"), ("shap", "ig"), ("lime", "ig")]
agg = {f"{a}_{b}": {"jaccard": [], "spearman": []} for a, b in pairs}
three_way, by_kind = [], {"correct": [], "error": []}
for rec in local_records:
    tk = {m: [t["word"] for t in rec["topk"][m]] for m in ("shap", "lime", "ig")}
    for a, b in pairs:
        agg[f"{a}_{b}"]["jaccard"].append(jaccard(tk[a], tk[b]))
        agg[f"{a}_{b}"]["spearman"].append(_spear(np.abs(rec[a]), np.abs(rec[b])))
    inter = set(tk["shap"]) & set(tk["lime"]) & set(tk["ig"])
    union = set(tk["shap"]) | set(tk["lime"]) | set(tk["ig"])
    tw = len(inter) / len(union) if union else 0.0
    three_way.append(tw); by_kind[rec["kind"]].append(tw)

def ms(v):
    return {"mean": round(float(np.mean(v)), 4), "sd": round(float(np.std(v)), 4), "n": len(v)} if v else None

agreement = {
    "part": "stage4_part4.1_local_explanations", "version": PART_VERSION,
    "principle": "P1 (no human); P3 (frozen TEST); P4 (rank/Jaccard geometry);  multi-method agreement",
    "methods": ["SHAP (sampling KernelSHAP)", "LIME (exp-kernel sparse linear)", "IG (32-step, zero baseline)"],
    "n_instances": len(local_records), "topk": TOPK,
    "pairwise": {k: {"jaccard": ms(v["jaccard"]), "spearman": ms(v["spearman"])} for k, v in agg.items()},
    "three_way_jaccard": ms(three_way),
    "three_way_by_kind": {k: ms(v) for k, v in by_kind.items()},
    "budgets": {"shap_samples": SHAP_SAMPLES, "lime_samples": LIME_SAMPLES, "ig_steps": IG_STEPS,
                "max_words": MAX_WORDS},
    "elapsed_seconds": elapsed,
    "honest_note": "Pilot-scale model on a small frozen TEST (Crisis scarce, 13 posts). This VALIDATES the "
                   "multi-method attribution machinery on the actual deployed model (carry-in invariant 1); "
                   "trigger CONTENT sharpens at the deferred data scale-up. SHAP/LIME perturb the full text "
                   "function (both backbones re-embedded); IG attributes the MuRIL sub-path, which is why "
                   "cross-method agreement is informative rather than tautological. IG magnitudes are ~1e-9 "
                   "(gradients through the frozen mean-pooled MuRIL backbone; fp32==fp64 verified -> real "
                   "signal, not noise); attributions are stored to significant figures (not fixed decimals) "
                   "and agreement uses RANK geometry, so the tiny IG scale does not affect the comparison.",
}
print("=== CROSS-METHOD AGREEMENT (mean top-%d Jaccard / Spearman of |attr|) ===" % TOPK)
for k in agg:
    j = agreement["pairwise"][k]["jaccard"]; s = agreement["pairwise"][k]["spearman"]
    print(f"  {k:10} Jaccard={j['mean']:.3f}+-{j['sd']:.3f}  Spearman={s['mean']:.3f}+-{s['sd']:.3f}")
tw = agreement["three_way_jaccard"]
print(f"  3-way     Jaccard={tw['mean']:.3f}+-{tw['sd']:.3f} (correct={agreement['three_way_by_kind']['correct']}, "
      f"error={agreement['three_way_by_kind']['error']})")

## 12. Persist local explanations + agreement

In [ ]:
with open(OUTPUT_DIR / "local_explanations.jsonl", "w", encoding="utf-8") as fh:
    for rec in local_records:
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
(OUTPUT_DIR / "method_agreement.json").write_text(json.dumps(agreement, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote local_explanations.jsonl (%d rows) + method_agreement.json", len(local_records))

## 13. Figures — per-instance attribution + agreement summary

In [ ]:
if HAS_MPL:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    def safe_label(w):
        try:
            w.encode("latin-1"); return w
        except Exception:
            return "<dev>"  # non-latin token -> placeholder (font-robust)

    n_fig = min(N_HEATMAP_FIGS, len(local_records))
    for rec in local_records[:n_fig]:
        words = rec["words"]
        order = np.argsort(-np.abs(np.array(rec["shap"])))[:TOPK]
        labels = [safe_label(words[i]) for i in order]
        # normalise each method to unit max-|attr| (IG ~1e-9 vs SHAP ~1e-2 would be invisible otherwise);
        # bars show SIGNED, per-method-normalised contribution (ranking is the comparable quantity)
        def _norm(v):
            v = np.array(v, float); mx = np.abs(v).max()
            return v / mx if mx > 0 else v
        sh_n, li_n, ig_n = _norm(rec["shap"]), _norm(rec["lime"]), _norm(rec["ig"])
        x = np.arange(len(order)); width = 0.26
        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.bar(x - width, [sh_n[i] for i in order], width, label="SHAP")
        ax.bar(x,         [li_n[i] for i in order], width, label="LIME")
        ax.bar(x + width, [ig_n[i] for i in order], width, label="IG")
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("normalised attribution to P(%s)" % TIER_NAMES[rec["target_explained"]])
        ax.set_title("rid=%s  true=%s pred=%s (%s, conf=%.2f)" % (
            rec["record_id"][:10], TIER_NAMES[rec["tier_true"]], TIER_NAMES[rec["tier_pred"]],
            rec["kind"], rec["pred_confidence"]), fontsize=9)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f"attribution_{rec['record_id'][:12]}.png", dpi=130)
        plt.close(fig)

    # agreement summary bar
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    keys = list(agg.keys()) + ["3-way"]
    jac = [agreement["pairwise"][k]["jaccard"]["mean"] for k in agg] + [agreement["three_way_jaccard"]["mean"]]
    spr = [agreement["pairwise"][k]["spearman"]["mean"] for k in agg] + [np.nan]
    x = np.arange(len(keys)); width = 0.38
    ax.bar(x - width/2, jac, width, label="top-%d Jaccard" % TOPK)
    ax.bar(x + width/2, spr, width, label="Spearman |attr|")
    ax.set_xticks(x); ax.set_xticklabels(keys, rotation=20, ha="right", fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel("agreement"); ax.legend(fontsize=8)
    ax.set_title("Cross-method attribution agreement (n=%d)" % len(local_records), fontsize=9)
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / "method_agreement.png", dpi=130); plt.close(fig)
    log.info("wrote %d attribution figures + method_agreement.png", n_fig)
else:
    log.warning("matplotlib absent — skipped figures")

## 14. This step is complete when

**notebook 11 is complete when:** SHAP, LIME, and IG are each **reproduced from a real run** on the **frozen TEST**
(P3) over the actual deployed model (faithfulness cosine asserted > 0.999); `local_explanations.jsonl` holds
per-instance token attributions + per-method top-k triggers; `method_agreement.json` reports pairwise + 3-way
top-k Jaccard and Spearman of |attribution|; sample `attribution_*.png` + `method_agreement.png` rendered; the
logged instance N and the pilot-scale caveat are written. **No human / no expert (P1 / ).**

In [ ]:
dod = all([
    (OUTPUT_DIR / "local_explanations.jsonl").exists(),
    (OUTPUT_DIR / "method_agreement.json").exists(),
    len(local_records) > 0,
    all(("shap" in r and "lime" in r and "ig" in r) for r in local_records),
])
print("=" * 64); print("Objective 4 · notebook 11 — LOCAL EXPLANATIONS CARD"); print("=" * 64)
print("methods        : SHAP (KernelSHAP) + LIME (exp-kernel ridge) + IG (32-step, zero baseline) — from scratch")
print("instances      :", len(local_records), "(curated: top-conf correct/tier + errors, D3)")
print("faithfulness   : re-embed vs stored MuRIL min cosine = %.5f (> 0.999)" % min_cos)
tw = agreement["three_way_jaccard"]
print("3-way agreement: top-%d Jaccard = %.3f +- %.3f" % (TOPK, tw["mean"], tw["sd"]))
print("outputs        : local_explanations.jsonl, method_agreement.json, attribution_*.png, method_agreement.png")
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(pilot-scale; machinery validated, content provisional)")